# Single Anderson impurity perturned by a gaussian light pulse

In [ ]:
include("../src/model.jl")

using SpecialFunctions: erf, erfc
using LinearAlgebra: diag, norm

using Plots, LaTeXStrings
default(fmt = :png)

In [ ]:
model = PhotonAssitedModel(
  #= SIAM parameters =#
  V₀=sqrt(0.15),
  ε₀f=-0.3,
  σ=0.3,
  
  #= Light-field parameters =#
  Γ₀=0.0,  # light-matter coupling
  ω₀=1e-1, # photon-ground state energy
  δ₀=1e-1, # photon artificial width
  
  #= Photon-pulse parameters =#
  tc=15,   # peak arrival time
  Ω=1e0,  # frequency bandwitdh of the pulse
  
  #= SLH photon formulation =#
  Γ=5.     # pump and loss strength
)

# Pulse and SLH shape
model.ξ = t -> (model.Ω^2 / (2π))^(1/4)* exp(-(t - model.tc)^2 * (model.Ω^2 / 4))
model.γ = t -> model.Γ * model.ξ(t)
model.λ = t -> model.Γ * model.ξ(t) / sqrt(0.5 * erfc((t - model.tc) * model.Ω / sqrt(2)))
model.hᵧ= t -> (model.ω₀ - 0.5im * (model.λ(t) + model.γ(t)))

data = let β=1e2, nᵞ = 0.0 # initial photon occupation
  ProblemData(model, β, NCA.LinearMesh(-20.0, 20.0, 2^15 - 1), nᵞ)
end;

plot(data.freqs.mesh, data.Aᵇ˹[1] |> imag, label=L"$A_b$")
plot!(data.freqs.mesh, data.Aᶠ˹[1] |> imag, label=L"$A_f$")
plot!(data.freqs.mesh, data.Aᶜ˹[1] |> imag, label=L"$A_c$")
title!("Tilde spectral functions")
xlabel!(L"$\omega$")

In [ ]:
# Initial condition
u0 = tuple(
  [data.GᶜL,data.GᶜG,data.GᶠL,data.GᶠG,data.GᵇL,data.GᵇG,data.GᵞL,data.GᵞG],
  [data.Aᶜ˹,data.GᶠT,data.GᶠM,data.GᵇT,data.GᵇM]
)

# Vertical step
f_vert(u, times, t1::Int, t2::Int) = f_vert_time(model, data, times, t1, t2)
f_vert(u, v, times, t1::Int, t2::Int) = f_vert(u, times, t1, t2) .+ f_vert_freq(model, data, times, t1, t2)

# Diagonal step
f_diag(u, times, t1::Int) = (println("t: $(times[t1])"); f_diag_time(model, data, times, t1))
f_diag(u, v, times, t1::Int) = f_diag(u, times, t1) .+ update_norms!(f_diag_freq(model, data, times, t1))

# Vertical step for mixed functions
f_line(u, v, times, t1::Int) = f_line_freq(model, data, times, t1)

thermal_norms = float(Inf)
update_norms!(n) = (global thermal_norms; thermal_norms = norm.(n); n)

"""
Step until the thermal contribution is small and then discard it
"""

# Criteria for evaluating the thermal collision integrals
# TODO: Accumulate norms over horizontal step (don't take into account just the diagonal)
thermal_criteria(norms) = max(norms...) < 1e-8

tspan = (0.0, 10.0)

# The solver will stop when tspan[2] is reached or `stop` evaluates to `false`
state, caches = kbsolve(f_vert, f_diag, u0, tspan[1], tspan[2]; f_line=f_line,
    init_dt=1e-3, atol=1e-6, rtol=1e-4, max_dt=0.5,
    stop=()->thermal_criteria(thermal_norms),
    w_line! =(t1)->calculate_Σ_freq!(model,data,t1),
    w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

@testset "KMS boundary conditions" begin
  @unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Aᶜ˹,GᶠT,GᶠM,GᵇT,GᵇM,ΣᶠT,ΣᵇT,freqs,fermi,fermi⁻ = data
  
  # G^R(t,t) = -i
#   @test all(diag(GᶜG.data - GᶜL.data) .≈ -1.0im)
  @test all(diag(GᵞG.data - GᵞL.data) .≈ -1.0im)
  @test all(diag(GᶠG.data) .≈ -1.0im)
  @test all(diag(GᵇG.data) .≈ -1.0im)

  ∫dω(g, f) = integrate(freqs.mesh, g .* f, TrapezoidalEvenFast()) / (2π)
  ≅(a,b) = isapprox(a,b;atol=5e-3)
  
  # C^<(t,0) = C˥(t,0+)
  @test all(GᶜL[t,1] ≅ ∫dω(fermi, Aᶜ˹[t]) for t in 1:size(Aᶜ˹,2))
  @test all(GᶠG[1,t] ≅ -∫dω(fermi⁻, GᶠT[t]) for t in 1:size(GᶠT,2))
  @test all(GᵇG[1,t] ≅ -∫dω(fermi⁻, GᵇT[t]) for t in 1:size(GᵇT,2))
  
  @test all(data.ΣᶠG[1,t] ≅ -∫dω(fermi⁻, data.ΣᶠT[t]) for t in 1:size(ΣᶠT,2))
  @test all(data.ΣᵇG[1,t] ≅ -∫dω(fermi⁻, data.ΣᵇT[t]) for t in 1:size(ΣᵇT,2))
end

# If we don't need to evaluate the thermal collision integrals anymore
if thermal_criteria(thermal_norms)
  
  # We need a new integrator-state without the mixed functions
  state = KadanoffBaym.VCABMState((state.u[1],), state.t, state.dt)
  
  state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
    atol=1e-6, rtol=1e-4, max_dt=1.0,
    w_time! = (t1,t2) -> calculate_Σ_time!(model,data,t1,t2),
    st_ch = (state, caches))
end

# The solver will stop when tspan[2] is reached or `stop` evaluates to `false`
# state, caches = kbsolve(f_vert, f_diag, u0[1], tspan[1], tspan[2];
#     init_dt=1e-3, atol=1e-7, rtol=1e-5, max_dt=0.5,
#     w_time! =(t1,t2)->calculate_Σ_time!(model,data,t1,t2))

In [ ]:
# Extract info
@unpack GᶜL,GᶜG,GᶠL,GᶠG,GᵇL,GᵇG,GᵞL,GᵞG,Aᶜ˹,GᶠT,GᶠM,GᵇT,GᵇM,ΣᶠT,ΣᵇT,freqs,fermi,fermi⁻ = data
times = state.t;

# Diagonostics
∫dω(g, f) = integrate(data.freqs.mesh, g .* f, TrapezoidalEvenFast()) / (2π)
∫dω(f) = integrate(data.freqs.mesh, f, TrapezoidalEvenFast()) / (2π)

p1 = plot(times, imag([2GᶠL[t,t] - GᵇL[t,t] for t in eachindex(times)]), xlabel=L"t", ylabel=L"Q")

p2 = plot(times, [imag(GᶠL[t,t]) for t in eachindex(times)], xlabel=L"t", ylabel=L"n", label=L"f")
p2 = plot!(times, [imag(-GᵇL[t,t]) for t in eachindex(times)], label=L"b")

p3 = plot(times, GᵞL.data |> diag .|> imag .|> -, label="Lesser", xlabel=L"t")
p3 = plot!(times, GᵞG.data |> diag .|> imag .|> -, label="Greater")
p3 = plot!(twinx(), times, model.ξ.(times), color=:green, legend=:right, label="Pulse", line=:dot)

display(p1)
display(p2)
display(p3)